# 06 — Yoko Musanga × Google Satellite Embedding regression

## Question
Can the **mean UAV-derived Musanga probability at 10 m** be predicted from the 64-dimensional Google Satellite Embedding for the Yoko study site?

This is deliberately a **single-site proof of concept**. It uses a random train/test split for rapid model development. Cross-site validation should be added later when the other UAV Musanga probability products are introduced.

## Workflow
1. Load the validated UAV `musanga_probability_uint16.tif`.
2. Download the **2020 Google Satellite Embedding V1** over the UAV footprint.
3. Use the embedding raster as the authoritative satellite grid.
4. Aggregate UAV probability onto that grid using **average resampling** — no Musanga threshold.
5. Build a table containing `musanga_prob_mean` and A00…A63.
6. Benchmark Random Forest and HistGradientBoosting regressors.
7. Export observed, predicted, and residual 10-m GeoTIFFs.

The target remains continuous throughout.

## BLOCK 01 — Install and import packages

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 01 — INSTALL AND IMPORT PACKAGES")
print("=" * 100)

!pip -q install earthengine-api geemap rasterio pyproj scikit-learn scipy joblib

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import ee
import geemap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
print("Imports OK")

## BLOCK 02 — Mount Google Drive and define paths

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 02 — MOUNT DRIVE AND DEFINE PATHS")
print("=" * 100)

from google.colab import drive
DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

PROJECT_ROOT = MYDRIVE / "Colab Notebooks" / "Musanga"
WORK_DIR = PROJECT_ROOT / "work"
INPUT_PROB = WORK_DIR / "whole_orthomosaic_inference" / "musanga_probability_uint16.tif"

OUT_DIR = WORK_DIR / "yoko_satellite_embedding"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_TIF = OUT_DIR / "Yoko_GoogleSatelliteEmbedding_2020_10m.tif"
OBS_TIF = OUT_DIR / "Yoko_UAV_Musanga_mean_10m.tif"
PRED_TIF = OUT_DIR / "Yoko_Embedding_Musanga_predicted_10m.tif"
RESID_TIF = OUT_DIR / "Yoko_Embedding_Musanga_residual_10m.tif"
TABLE_CSV = OUT_DIR / "Yoko_embedding_training_table.csv"
MODEL_FILE = OUT_DIR / "Yoko_best_embedding_regressor.joblib"
SUMMARY_JSON = OUT_DIR / "Yoko_embedding_regression_summary.json"

assert INPUT_PROB.exists(), f"Missing input: {INPUT_PROB}"
print("INPUT:", INPUT_PROB)
print("OUTPUT:", OUT_DIR)

## BLOCK 03 — Configuration and inspect UAV probability raster

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 03 — CONFIGURATION / UAV INPUT")
print("=" * 100)

EE_PROJECT = "tropsedslu"
EMBEDDING_YEAR = 2020
MIN_UAV_COVERAGE = 0.90
TEST_SIZE = 0.20
FORCE_DOWNLOAD_EMBEDDINGS = False

with rasterio.open(INPUT_PROB) as src:
    print("CRS:", src.crs)
    print("Shape:", src.height, src.width)
    print("Resolution:", src.res)
    print("dtype:", src.dtypes[0], "nodata:", src.nodata)
    print("bounds:", src.bounds)

    # Sparse sample only: do not load the full UAV raster.
    oh = min(1000, src.height)
    ow = min(1000, src.width)
    sample = src.read(1, out_shape=(oh, ow), resampling=Resampling.nearest).astype("float32")
    if src.nodata is not None:
        sample = sample[sample != src.nodata]
    sample = sample[np.isfinite(sample)]

print("Sample min/max:", float(sample.min()), float(sample.max()))
PROB_SCALE = 65535.0 if sample.max() > 1.5 else 1.0
print("Probability divisor:", PROB_SCALE)
print("Embedding year:", EMBEDDING_YEAR)

## BLOCK 04 — Authenticate Earth Engine and download the 2020 embedding

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 04 — EARTH ENGINE / GOOGLE SATELLITE EMBEDDING")
print("=" * 100)

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

with rasterio.open(INPUT_PROB) as src:
    b = src.bounds
    left, bottom, right, top = transform_bounds(
        src.crs, "EPSG:4326", b.left, b.bottom, b.right, b.top, densify_pts=21
    )

roi = ee.Geometry.Rectangle([left, bottom, right, top], geodesic=False)

collection = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f"{EMBEDDING_YEAR}-01-01", f"{EMBEDDING_YEAR + 1}-01-01")
    .filterBounds(roi)
)
print("Embedding images intersecting Yoko:", collection.size().getInfo())

band_names = [f"A{i:02d}" for i in range(64)]
embedding = collection.mosaic().select(band_names)

if FORCE_DOWNLOAD_EMBEDDINGS or not EMBED_TIF.exists():
    print("Downloading 64-band embedding raster...")
    geemap.ee_export_image(
        embedding,
        filename=str(EMBED_TIF),
        scale=10,
        region=roi,
        file_per_band=False,
    )
else:
    print("Reusing:", EMBED_TIF)

assert EMBED_TIF.exists(), "Embedding export failed."
with rasterio.open(EMBED_TIF) as src:
    print("Embedding CRS:", src.crs)
    print("Embedding shape:", src.height, src.width)
    print("Embedding bands:", src.count)
    print("Embedding resolution:", src.res)
    assert src.count == 64, f"Expected 64 bands, got {src.count}"

## BLOCK 05 — Aggregate UAV probability directly to the embedding grid

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 05 — UAV MEAN PROBABILITY ON EMBEDDING GRID")
print("=" * 100)

with rasterio.open(EMBED_TIF) as emb:
    dst_crs = emb.crs
    dst_transform = emb.transform
    dst_shape = (emb.height, emb.width)
    out_profile = emb.profile.copy()

# Reproject directly from the source raster. Rasterio/GDAL performs average
# resampling without us loading the ~1-billion-pixel UAV raster into RAM.
with rasterio.open(INPUT_PROB) as src:
    obs_raw = np.full(dst_shape, np.nan, dtype="float32")
    coverage = np.zeros(dst_shape, dtype="float32")

    reproject(
        source=rasterio.band(src, 1),
        destination=obs_raw,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        dst_nodata=np.nan,
        resampling=Resampling.average,
    )

    # Coverage from a temporary source-sized mask would be expensive.
    # Reproject a GDAL mask band instead; 255 = valid, 0 = invalid.
    src_mask = src.dataset_mask()
    reproject(
        source=src_mask,
        destination=coverage,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=0,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        dst_nodata=0,
        resampling=Resampling.average,
    )

coverage /= 255.0
obs = np.clip(obs_raw / PROB_SCALE, 0.0, 1.0)
good = np.isfinite(obs) & (coverage >= MIN_UAV_COVERAGE)

print("10-m cells:", obs.size)
print(f"Cells with >= {MIN_UAV_COVERAGE:.0%} UAV coverage:", int(good.sum()))
print("Mean target:", float(np.nanmean(obs[good])))
print("Median target:", float(np.nanmedian(obs[good])))
print("P90 target:", float(np.nanpercentile(obs[good], 90)))
print("Max target:", float(np.nanmax(obs[good])))

profile1 = out_profile.copy()
profile1.update(count=1, dtype="float32", nodata=-9999.0, compress="deflate")
with rasterio.open(OBS_TIF, "w", **profile1) as dst:
    dst.write(np.where(good, obs, -9999.0).astype("float32"), 1)

print("Saved:", OBS_TIF)

## BLOCK 06 — Inspect target distribution

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 06 — TARGET DISTRIBUTION")
print("=" * 100)

y_all = obs[good]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_all, bins=50)
axes[0].set_xlabel("Mean UAV Musanga probability at 10 m")
axes[0].set_ylabel("Number of cells")
axes[0].set_title("Full distribution")

positive_view = y_all[y_all > 0.001]
if len(positive_view):
    axes[1].hist(positive_view, bins=50)
axes[1].set_xlabel("Mean UAV Musanga probability")
axes[1].set_title("Cells > 0.001 (display only)")
plt.tight_layout()
plt.show()

for q in [0, .25, .5, .75, .9, .95, .99, 1]:
    print(f"q{q:>4}: {np.quantile(y_all, q):.6f}")

## BLOCK 07 — Build canonical 10-m training table

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 07 — BUILD TRAINING TABLE")
print("=" * 100)

with rasterio.open(EMBED_TIF) as src:
    Xcube = src.read().astype("float32")
    emb_nodata = src.nodata
    transform = src.transform

emb_valid = np.all(np.isfinite(Xcube), axis=0)
if emb_nodata is not None:
    emb_valid &= np.all(Xcube != emb_nodata, axis=0)

mask = good & emb_valid
rows, cols = np.where(mask)
X = Xcube[:, mask].T
target = obs[mask]
cov = coverage[mask]

xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
df = pd.DataFrame({
    "cell_id": np.arange(len(target), dtype=int),
    "x": np.asarray(xs),
    "y": np.asarray(ys),
    "uav_coverage": cov,
    "musanga_prob_mean": target,
})
for i in range(64):
    df[f"A{i:02d}"] = X[:, i]

df.to_csv(TABLE_CSV, index=False)
print("Rows:", len(df))
print("Predictors:", X.shape[1])
print("Saved:", TABLE_CSV)
display(df.head())

## BLOCK 08 — Random Yoko train/test benchmark

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08 — RANDOM TRAIN/TEST BENCHMARK")
print("=" * 100)

idx = np.arange(len(target))
train_idx, test_idx = train_test_split(
    idx, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, max_features="sqrt",
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, l2_regularization=1.0,
        random_state=RANDOM_SEED
    ),
}

results, fitted = [], {}
for name, model in models.items():
    model.fit(X[train_idx], target[train_idx])
    pred = np.clip(model.predict(X[test_idx]), 0, 1)
    results.append({
        "model": name,
        "RMSE": mean_squared_error(target[test_idx], pred) ** 0.5,
        "MAE": mean_absolute_error(target[test_idx], pred),
        "R2": r2_score(target[test_idx], pred),
        "Spearman_rho": spearmanr(target[test_idx], pred).statistic,
    })
    fitted[name] = model

baseline = np.full(len(test_idx), target[train_idx].mean())
results.append({
    "model": "Mean_baseline",
    "RMSE": mean_squared_error(target[test_idx], baseline) ** 0.5,
    "MAE": mean_absolute_error(target[test_idx], baseline),
    "R2": r2_score(target[test_idx], baseline),
    "Spearman_rho": np.nan,
})

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

candidate_results = results_df[results_df["model"] != "Mean_baseline"]
best_name = candidate_results.iloc[0]["model"]
best_model = fitted[best_name]
print("Selected model:", best_name)

## BLOCK 09 — Held-out diagnostic plots

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 09 — HELD-OUT DIAGNOSTICS")
print("=" * 100)

test_pred = np.clip(best_model.predict(X[test_idx]), 0, 1)
test_obs = target[test_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(test_obs, test_pred, s=8, alpha=0.35)
lim = max(float(test_obs.max()), float(test_pred.max()))
axes[0].plot([0, lim], [0, lim], "--")
axes[0].set_xlabel("Observed UAV mean probability")
axes[0].set_ylabel("Predicted probability")
axes[0].set_title(f"Held-out cells — {best_name}")

res = test_obs - test_pred
axes[1].scatter(test_pred, res, s=8, alpha=0.35)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Observed − predicted")
axes[1].set_title("Residuals")
plt.tight_layout()
plt.show()

## BLOCK 10 — Refit on all Yoko cells and export maps

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 10 — REFIT / EXPORT MAPS")
print("=" * 100)

final_model = models[best_name]
final_model.fit(X, target)
joblib.dump(final_model, MODEL_FILE)

pred_all = np.clip(final_model.predict(X), 0, 1).astype("float32")
pred_grid = np.full(dst_shape, -9999.0, dtype="float32")
resid_grid = np.full(dst_shape, -9999.0, dtype="float32")
pred_grid[rows, cols] = pred_all
resid_grid[rows, cols] = target - pred_all

for path, arr in [(PRED_TIF, pred_grid), (RESID_TIF, resid_grid)]:
    with rasterio.open(path, "w", **profile1) as dst:
        dst.write(arr, 1)

summary = {
    "site": "Yoko",
    "embedding_year": EMBEDDING_YEAR,
    "target": "mean UAV-derived Musanga probability on Google embedding grid",
    "threshold_used_for_target": False,
    "minimum_uav_coverage": MIN_UAV_COVERAGE,
    "n_cells": int(len(target)),
    "test_fraction": TEST_SIZE,
    "split": "random single-site development split",
    "selected_model": best_name,
    "benchmark": results_df.replace({np.nan: None}).to_dict(orient="records"),
    "outputs": {
        "observed_10m": str(OBS_TIF),
        "predicted_10m": str(PRED_TIF),
        "residual_10m": str(RESID_TIF),
        "training_table": str(TABLE_CSV),
        "model": str(MODEL_FILE),
    },
}
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved prediction:", PRED_TIF)
print("Saved residual:", RESID_TIF)
print("Saved summary:", SUMMARY_JSON)

## BLOCK 11 — Interpretation and next step

This notebook is a **Yoko development test**, not yet a test of geographic transferability.

The random held-out metrics test whether the 2020 Google Satellite Embedding contains enough local information to reconstruct variation in the UAV-derived Musanga signal at Yoko. Nearby pixels are spatially related, so these metrics should not be interpreted as independent regional validation.

The stronger next experiment is **cross-site transfer**: train at Yoko and predict an independent site with its own UAV Musanga probability raster.

If the embedding-only experiment is promising, the next benchmark should compare:
- Google Satellite Embedding only;
- explicit Sentinel-2 temporal/spectral features;
- Satellite Embedding + Sentinel-2 temporal features.